# Limpieza de datos

Objetivo: cargar el dataset oficial desde `data/raw`, auditar calidad de datos con criterio `ds-dq` y aplicar una imputacion por mediana defendible para no perder aproximadamente un tercio de la base con `dropna()`.

Regla de trabajo: primero caracterizamos, despues intervenimos.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = "../data/raw/datos.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,160
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,121
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,130


## 1. Diagnostico estructural

Revisamos dimensiones, tipos y memoria. Este paso evita limpiar a ciegas.

In [3]:
print("Shape:", df.shape)
df.info(memory_usage="deep")

Shape: (5630, 20)
<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAdd

In [4]:
df.dtypes.to_frame("dtype")

,dtype
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


## 2. Valores faltantes

Primero identificamos cuantos nulos hay por columna y que porcentaje representan.

In [5]:
missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

missing_summary

,missing_count,missing_pct
DaySinceLastOrder,307,5.45
OrderAmountHikeFromlastYear,265,4.71
Tenure,264,4.69
OrderCount,258,4.58
CouponUsed,256,4.55
HourSpendOnApp,255,4.53
WarehouseToHome,251,4.46


In [6]:
rows_with_missing = df.isna().any(axis=1).sum()
rows_with_missing_pct = rows_with_missing / len(df) * 100

print(f"Filas con al menos un faltante: {rows_with_missing} ({rows_with_missing_pct:.2f}%)")
print(f"Filas que quedarian con dropna(): {len(df.dropna())} de {len(df)}")

Filas con al menos un faltante: 1856 (32.97%)
Filas que quedarian con dropna(): 3774 de 5630


## 3. Duplicados y clave de negocio

`CustomerID` funciona como identificador. Revisamos duplicados exactos, IDs repetidos y perfiles idénticos al excluir el identificador.

In [7]:
profile_cols = [col for col in df.columns if col != "CustomerID"]

print("Duplicados exactos:", df.duplicated().sum())
print("CustomerID duplicados:", df.duplicated(subset=["CustomerID"]).sum())
print(
    "Perfiles duplicados sin considerar CustomerID:",
    df.duplicated(subset=profile_cols, keep="first").sum(),
)

Duplicados exactos: 0
CustomerID duplicados: 0
Perfiles duplicados sin considerar CustomerID: 556


## 4. Cardinalidad y variables categoricas

Revisamos cardinalidad para detectar categorias raras, errores de tipeo o variables que requieran encoding posterior.

In [8]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
cardinality = df[categorical_cols].nunique(dropna=False).sort_values(ascending=False).to_frame("n_unique")
cardinality

C:\Users\binel\AppData\Local\Temp\ipykernel_35260\2286258193.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


,n_unique
PreferredPaymentMode,7
PreferedOrderCat,6
PreferredLoginDevice,3
MaritalStatus,3
Gender,2


In [9]:
for col in categorical_cols:
    print("\n" + col)
    print(df[col].value_counts(dropna=False).head(10))


PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

Gender
Gender
Male      3384
Female    2246
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64

MaritalStatus
MaritalStatus
Married     2986
Single      1796
Divorced     848
Name: count, dtype: int64


## 5. Decision de limpieza

No usamos `dropna()` porque elimina 1856 filas, es decir 32.97% de los clientes. En un problema de churn, esa perdida es demasiado grande para una limpieza inicial.

Usaremos imputacion por mediana para variables numericas con faltantes. Es una regla robusta a outliers, simple de explicar y consistente con el pipeline de modelado.

Eliminamos 556 perfiles exactamente duplicados al comparar todas las variables excepto `CustomerID`. Como el identificador era la unica diferencia, conservamos la primera aparicion de cada perfil mediante `drop_duplicates(..., keep='first')`. El dataset pasa de 5.630 a 5.074 registros.

Ademas, estandarizamos categorias equivalentes antes de guardar el dataset limpio: `COD` y `Cash on Delivery`, `CC` y `Credit Card`, `Phone` y `Mobile Phone`, `Mobile` y `Mobile Phone`.

In [10]:
df_clean = df.copy()

# Se conserva la primera fila de cada perfil idéntico y se ignora CustomerID,
# porque el identificador no representa una característica del cliente.
profile_cols = [col for col in df_clean.columns if col != "CustomerID"]
rows_before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=profile_cols, keep="first").copy()
rows_removed_as_duplicates = rows_before_dedup - len(df_clean)

print(f"Perfiles duplicados eliminados: {rows_removed_as_duplicates}")
print(f"Filas luego de deduplicar: {len(df_clean)}")

category_replacements = {
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card",
    },
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone",
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone",
    },
}

for col, replacements in category_replacements.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(replacements)


id_col = "CustomerID"
target_col = "Churn"

numeric_cols = df_clean.select_dtypes(include=["number"]).columns.tolist()
numeric_feature_cols = [col for col in numeric_cols if col not in [id_col, target_col]]

numeric_cols_with_missing = [col for col in numeric_feature_cols if df_clean[col].isna().any()]
numeric_cols_with_missing

Perfiles duplicados eliminados: 556
Filas luego de deduplicar: 5074


['Tenure',
 'WarehouseToHome',
 'HourSpendOnApp',
 'OrderAmountHikeFromlastYear',
 'CouponUsed',
 'OrderCount',
 'DaySinceLastOrder']

In [11]:
median_values = df_clean[numeric_feature_cols].median()
df_clean[numeric_feature_cols] = df_clean[numeric_feature_cols].fillna(median_values)

print("Medianas usadas para imputacion numerica:")
display(median_values.to_frame("mediana").round(2))


Medianas usadas para imputacion numerica:


,mediana
Tenure,9.0
CityTier,1.0
WarehouseToHome,13.0
HourSpendOnApp,3.0
NumberOfDeviceRegistered,4.0
SatisfactionScore,3.0
NumberOfAddress,3.0
Complain,0.0
OrderAmountHikeFromlastYear,15.0
CouponUsed,1.0


## 6. Ajustes posteriores a la imputacion

La mediana puede devolver decimales cuando la cantidad de observaciones es par. Para variables que representan conteos o cantidades enteras, redondeamos. Tambien acotamos variables naturalmente no negativas.

In [12]:
integer_like_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

for col in integer_like_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].round().astype(int)

non_negative_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
]

for col in non_negative_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].clip(lower=0)

df_clean.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4,Mobile Phone,3,6,Debit Card,Female,3,3,Laptop & Accessory,2,Single,9,1,11,1,1,5,160
1,50002,1,9,Mobile Phone,1,8,UPI,Male,3,4,Mobile Phone,3,Single,7,1,15,0,1,0,121
2,50003,1,9,Mobile Phone,1,30,Debit Card,Male,2,4,Mobile Phone,3,Single,6,1,14,0,1,3,120
3,50004,1,0,Mobile Phone,3,15,Debit Card,Male,2,4,Laptop & Accessory,5,Single,8,0,23,0,1,3,134
4,50005,1,0,Mobile Phone,1,12,Credit Card,Male,3,3,Mobile Phone,5,Single,3,0,11,1,1,3,130


## 7. Verificacion de cierre

Chequeamos nulos despues, dimensiones y tasa de churn. El target no se modifica; la tasa puede variar levemente porque se eliminan perfiles duplicados antes de guardar el dataset limpio.

In [13]:
df_clean.isna().sum().sort_values(ascending=False).head(15)

CustomerID                  0
Churn                       0
Tenure                      0
PreferredLoginDevice        0
CityTier                    0
WarehouseToHome             0
PreferredPaymentMode        0
Gender                      0
HourSpendOnApp              0
NumberOfDeviceRegistered    0
PreferedOrderCat            0
SatisfactionScore           0
MaritalStatus               0
NumberOfAddress             0
Complain                    0
dtype: int64

In [14]:
print("Shape original:", df.shape)
print("Shape limpio:", df_clean.shape)
print("Churn rate original:", round(df["Churn"].mean(), 4))
print("Churn rate limpio:", round(df_clean["Churn"].mean(), 4))
print(
    "Perfiles duplicados restantes según el criterio aplicado:",
    df_clean.duplicated(subset=profile_cols, keep="first").sum(),
)

Shape original: (5630, 20)
Shape limpio: (5074, 20)
Churn rate original: 0.1684
Churn rate limpio: 0.1657
Perfiles duplicados restantes según el criterio aplicado: 3


## 8. Guardado

Este archivo limpio sirve para EDA y documentacion de limpieza. Para modelado, la imputacion debe moverse dentro de un `Pipeline` y aplicarse despues del split train/test estratificado, para evitar leakage.

In [15]:
OUTPUT_PATH = "../data/processed/datos_limpios.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Archivo limpio guardado en: {OUTPUT_PATH}")

Archivo limpio guardado en: ../data/processed/datos_limpios.csv
